# 00 - Load Collection A -> merged TMT / iTRAQ / LabelFree (+ CCLE)

Imitates `05. validation/1. process_collection_a.ipynb` **for the loader only**: it groups Collection A
cohorts by platform, concatenates each group, maps RNA Ensembl IDs to gene symbols, filters >50%-missing
features/samples, transposes to samples x features, and saves per-platform RNA/protein matrices. It also
loads CCLE. Output feeds notebook `02_match_and_normalize`.

> Set `DATA_DIR` to your Collection A root and `CCLE_*` paths to your CCLE joblibs.

In [1]:
from pathlib import Path
import os, glob, sys, warnings
import numpy as np, pandas as pd, joblib
warnings.simplefilter("ignore", category=FutureWarning)

DATA_DIR   = Path("../data/Collection_A")          # <- Collection A cohorts (subfolders)
OUT_DIR    = Path("../data/loaded"); OUT_DIR.mkdir(parents=True, exist_ok=True)
CCLE_RNA   = Path("../data/CCLE_rna.joblib")        # samples x genes
CCLE_PROT  = Path("../data/CCLE_protein.joblib")    # samples x proteins
MAPPING_FILE = DATA_DIR / "brain" / "rows_metadata.csv"   # Ensembl -> symbol

COHORT_GROUPS = {
    "TMT":      ["colon_2019","renal_2019","Endomet_val","breast2020","pedbrain_2020",
                 "lung_2020_smoke","lung_val","lung2021","headneck_2021","pancrea2021",
                 "brain_2021","liver_val"],
    "iTRAQ":    ["breast_2016","ovary_2016","stomach_2019"],
    "LabelFree":["brain","colon","prostate","liver","lung"],
}

## 1. filter helper (same >50% rule as 05.validation)

In [2]:
def filter_nans(df, threshold=0.5):
    if df.empty: return df
    df2 = df.dropna(thresh=int(df.shape[1]*threshold), axis=0)
    df2 = df2.dropna(thresh=int(df2.shape[0]*threshold), axis=1)
    print(f"   dropped {df.shape[0]-df2.shape[0]} features, {df.shape[1]-df2.shape[1]} samples")
    return df2

## 2. Load + concatenate each platform group

In [3]:
group_data = {g: {"prot": [], "rna": []} for g in COHORT_GROUPS}
subdirs = [f.path for f in os.scandir(DATA_DIR) if f.is_dir()] if DATA_DIR.exists() else []
print(f"Found {len(subdirs)} cohort folders")
for subdir in subdirs:
    name = os.path.basename(subdir)
    grp = next((g for g,lst in COHORT_GROUPS.items() if name in lst), None)
    if grp is None: continue
    pf = glob.glob(os.path.join(subdir,"protein_matrix_*.txt"))
    rf = glob.glob(os.path.join(subdir,"RNA_matrix_*.txt"))
    if not (pf and rf): continue
    dp = pd.read_csv(pf[0], sep="\t", index_col=0)
    dr = pd.read_csv(rf[0], sep="\t", index_col=0)
    common = dp.columns.intersection(dr.columns)
    if len(common)==0: continue
    dp, dr = dp[common], dr[common]
    newnames = [f"{name.upper()}_{i+1}" for i in range(len(common))]
    dp.columns = newnames; dr.columns = newnames
    group_data[grp]["prot"].append(dp); group_data[grp]["rna"].append(dr)
print("cohorts loaded per group:", {g:len(d["prot"]) for g,d in group_data.items()})

Found 20 cohort folders
cohorts loaded per group: {'TMT': 12, 'iTRAQ': 3, 'LabelFree': 5}


## 3. Map RNA IDs, filter, transpose, save

In [4]:
id2sym = {}
if MAPPING_FILE.exists():
    m = pd.read_csv(MAPPING_FILE).dropna(subset=["ensembl_gene_id","gene_symbol"])
    id2sym = dict(zip(m["ensembl_gene_id"], m["gene_symbol"]))

for grp, dfs in group_data.items():
    if not dfs["prot"]:
        print(f"[skip] {grp}: no data"); continue
    prot = pd.concat(dfs["prot"], axis=1, join="outer", sort=True)
    rna  = pd.concat(dfs["rna"],  axis=1, join="outer", sort=True)
    if id2sym:
        # RNA rows: Ensembl -> symbol
        rna.index = rna.index.map(lambda x: id2sym.get(str(x).split(".")[0], id2sym.get(x, x)))
        if rna.index.duplicated().any(): rna = rna.groupby(level=0).mean()
        # PROTEIN rows: CPTAC protein tables are Ensembl gene IDs -> map to symbol too
        prot.index = prot.index.map(lambda x: id2sym.get(str(x).split(".")[0], id2sym.get(x, x)))
        if prot.index.duplicated().any(): prot = prot.groupby(level=0).mean()
    print(f"=== {grp} ===  raw prot {prot.shape} rna {rna.shape}")
    prot = filter_nans(prot); rna = filter_nans(rna)
    prot = prot.T.loc[lambda d: ~d.index.duplicated(keep="first")]
    rna  = rna.T.loc[lambda d: ~d.index.duplicated(keep="first")]
    joblib.dump(rna,  OUT_DIR / f"{grp}_rna.joblib", compress=3)
    joblib.dump(prot, OUT_DIR / f"{grp}_protein.joblib", compress=3)
    print(f"   saved {grp}: RNA {rna.shape}, Protein {prot.shape}")
joblib.dump(id2sym, OUT_DIR / "id2sym.joblib")
print(f"id2sym mapping: {len(id2sym)} entries -> {OUT_DIR/'id2sym.joblib'}")

=== TMT ===  raw prot (11259, 1924) rna (32840, 1924)
   dropped 4374 features, 0 samples
   dropped 17839 features, 0 samples
   saved TMT: RNA (1924, 15001), Protein (1924, 6885)
=== iTRAQ ===  raw prot (7569, 276) rna (18854, 276)
   dropped 2575 features, 0 samples
   dropped 4333 features, 0 samples
   saved iTRAQ: RNA (276, 14521), Protein (276, 4994)
=== LabelFree ===  raw prot (5767, 360) rna (25219, 360)
   dropped 3204 features, 0 samples
   dropped 10272 features, 0 samples
   saved LabelFree: RNA (360, 14947), Protein (360, 2563)
id2sym mapping: 52376 entries -> ..\data\loaded\id2sym.joblib


## 4. CCLE

In [5]:
if CCLE_RNA.exists() and CCLE_PROT.exists():
    Xc = joblib.load(CCLE_RNA); Yc = joblib.load(CCLE_PROT)
    Xc.columns = Xc.columns.map(str); Yc.columns = Yc.columns.map(str)
    joblib.dump(Xc, OUT_DIR / "CCLE_rna.joblib", compress=3)
    joblib.dump(Yc, OUT_DIR / "CCLE_protein.joblib", compress=3)
    print("CCLE:", Xc.shape, Yc.shape)
else:
    print("CCLE joblibs not found - set CCLE_RNA / CCLE_PROT. (Synthetic fallback used in smoke test.)")
print("\nLoaded matrices in", OUT_DIR, "-> feed notebook 02.")

CCLE: (369, 22614) (369, 5113)

Loaded matrices in ..\data\loaded -> feed notebook 02.
